# task_19 Solution

In [ ]:

from pathlib import Path
import pandas as pd


base = Path("../../tasks/task_19/data/")


In [ ]:

matches = pd.read_csv(base / "matches.csv")
events = pd.read_csv(base / "events.csv")

Q1: Using 3 points for a win, 1 for a draw, and 0 for a loss, which team_id finishes with the most points?

In [ ]:
goals = events[events["event_type"] == "goal"]
goals_by_match_team = goals.groupby(["match_id", "team_id"]).size().unstack(fill_value=0)
match_scores = matches.set_index("match_id").join(goals_by_match_team, how="left").fillna(0)
all_teams = set(matches["home_team"]) | set(matches["away_team"])
for team in all_teams:
    if team not in match_scores.columns:
        match_scores[team] = 0

points = {team: 0 for team in all_teams}
for row in match_scores.itertuples():
    home_goals = getattr(row, row.home_team)
    away_goals = getattr(row, row.away_team)
    if home_goals > away_goals:
        points[row.home_team] += 3
    elif away_goals > home_goals:
        points[row.away_team] += 3
    else:
        points[row.home_team] += 1
        points[row.away_team] += 1
q1 = sorted(points.items(), key=lambda x: (-x[1], x[0]))[0][0]
print(q1)


Q3: In what percentage of matches did the team leading at halftime fail to win the match, rounded to 1 decimal place?

In [ ]:
first_half_goals = goals[goals["minute"] <= 45]
halftime = first_half_goals.groupby(["match_id", "team_id"]).size().unstack(fill_value=0)
fail_count = 0
denom = 0
for row in matches.itertuples(index=False):
    h1 = halftime.loc[row.match_id, row.home_team] if row.match_id in halftime.index and row.home_team in halftime.columns else 0
    a1 = halftime.loc[row.match_id, row.away_team] if row.match_id in halftime.index and row.away_team in halftime.columns else 0
    if h1 == a1:
        continue
    denom += 1
    hg = match_scores.loc[row.match_id, row.home_team]
    ag = match_scores.loc[row.match_id, row.away_team]
    if h1 > a1 and not (hg > ag):
        fail_count += 1
    if a1 > h1 and not (ag > hg):
        fail_count += 1
q3 = f"{(100 * fail_count / denom):.1f}%"
print(q3)
